# Visual Asset Auditing System - Test Bench

This notebook implements the **Test Bench** tier of the Visual Asset Auditing System. It demonstrates:

1. Connecting to AlloyDB and GCS.
2. Running a Hybrid Search (pgvector + FTS).
3. Detecting the drop-off point and selecting the top 60 candidates (High Confidence + Borderline).
4. Running Gemini 3.5 Flash online inference to audit the selected assets.
5. Creating and updating the `audit_results` table in AlloyDB.

*Transcribed from IMG_9422.jpeg. Additional source screenshots will be added below in the order received.*

In [ ]:
# Install required libraries
!pip install -q google-genai google-cloud-vision google-cloud-aiplatform kfp google-cloud-pipeline-components pgvector asyncpg kneed pandas numpy pillow nest-asyncio sqlalchemy "protobuf<5.0.0dev"


## Package Installation

This cell installs all necessary external libraries (Google GenAI, Cloud Vision, AI Platform, pgvector, asyncio) to equip the notebook environment.

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

import google.genai as genai
from google.genai import types

print("Google GenAI SDK imported successfully.")


## Google Cloud Authentication

This cell handles authentication with Google Cloud using Colab auth utilities, sets up API project client.

In [ ]:
# AlloyDB Connection Setup using SQLAlchemy Pool + AsyncConnector
import asyncio
import asyncpg
from typing import Tuple
from sqlalchemy.ext.asyncio import create_async_engine, AsyncEngine
from google.cloud.alloydb.connector import IPTypes, AsyncConnector

_engine_cache = {}
_connector_cache = {}

async def get_alloydb_connection(reuse: bool = True) -> Tuple[AsyncEngine, AsyncConnector]:
    """Establishes and pools AlloyDB connections using SQLAlchemy and the AsyncConnector, with automatic timeout diagnostics."""
    global _engine_cache
    global _connector_cache

    if reuse and 'default' in _engine_cache:
        return _engine_cache['default'], _connector_cache['default']

    # Use lazy refresh for serverless/Colab environments
    connector = AsyncConnector(refresh_strategy="lazy")

    async def getconn():
        # Handle case where user pasted the full resource path or just the instance ID
        instance_uri = ALLOYDB_INSTANCE
        if not instance_uri.startswith("projects/"):
            instance_uri = f"projects/{PROJECT_ID}/locations/{REGION}/clusters/{ALLOYDB_CLUSTER}/instances/{ALLOYDB_INSTANCE}"

        try:
            # Enforce 10-second connection timeout to prevent hanging loop CancelledErrors
            conn = await asyncio.wait_for(
                connector.connect(
                    instance_uri,
                    "asyncpg",
                    user=DB_USER,
                    password=DB_PASSWORD,
                    db=DB_NAME,
                    enable_iam_auth=False, # Set to True if using IAM auth
                    ip_type=IPTypes.PUBLIC # Adjust to PUBLIC or PRIVATE
                ),
                timeout=10.0
            )
            return conn
        except asyncio.TimeoutError:
            raise ConnectionError(
                f"AlloyDB connection timed out (10s) to {instance_uri}. "
                "Ensure your client IP is authorized in the AlloyDB Public IP console, "
                "or check your VPC network access if running internally."
            )
        except Exception as e:
            raise ConnectionError(f"Failed to connect to AlloyDB: {e}")

    engine = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        echo=False,
        pool_size=10,
        max_overflow=20,
        pool_pre_ping=True # Force SQLAlchemy to health-check connections
    )

    if reuse:
        _engine_cache['default'] = engine
        _connector_cache['default'] = connector

    return engine, connector


In [ ]:
# # Run the setup
# import nest_asyncio
# nest_asyncio.apply()
# try:
#     asyncio.run(setup_database())
# except Exception as e:
#     print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# Database Connection Verification (Read-Only Check – No DDL or Schema Changes)
import asyncio
import nest_asyncio

async def verify_database_connection():
    """Verifies connection to AlloyDB and confirms the visual_assets table is reachable without making any DDL changes."""
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            count = await db.fetchval(f"SELECT COUNT(*) FROM {DB_SCHEMA}.visual_assets;")
            print(f"Connected to AlloyDB successfully. Schema '{DB_SCHEMA}.visual_assets' is ready ({count} assets found).")
    except Exception as e:
        print(f"Database connection status: {e}")

# Run connection check
nest_asyncio.apply()
try:
    asyncio.run(verify_database_connection())
except Exception as e:
    print(f"Skipping execution: Database connection not configured yet ({e})")


In [ ]:
# 1. Audit Config Generator & Embedding Generation
import json
import asyncio
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, List, Dict, Tuple
from pydantic import BaseModel, Field

# Shared executor for query-side embedding calls (avoids spinning up a new pool per request)
_EMBED_EXECUTOR = ThreadPoolExecutor(max_workers=8)

# Define Pydantic model for the structured Audit Context.
class AuditContextModel(BaseModel):
    audit_goal: str = Field(description="Refined, precise version of the user's goal.")
    image_description: Optional[str] = Field(None, description="Exhaustive forensic description of the reference image if provided (target brand & scope, asset category, exact visual signatures, color/contrast styling, compliance classification), else null.")
    reference_is_composite_canvas: bool = Field(description="True if the reference image is a composite layout, webpage screenshot, hero banner, or real-world photograph containing the logo/asset. False if the reference image represents an isolated, standalone logo on a plain background.")
    match_mode: str = Field(description="One of 'exact', 'similar', 'discovery'. 'exact' = the user wants THIS exact image/asset: same composition and subject (different scale, compression, resolution, or being embedded as a small element inside a larger canvas is still a match; different crops, poses, styles, or variants of the subject are NOT). 'similar' = the user wants the target subject in ANY variation: cropped/partial views (e.g. only the mascot's head), blurred, low-resolution, recolored, resized, or embedded instances. 'discovery' = all iterations/versions of the brand/subject regardless of state. Infer from the user's wording ('this exact image' -> exact; 'similar images', 'like this' -> similar; 'all assets of X' -> discovery); default to 'similar' when ambiguous.")
    inclusion_criteria: List[str] = Field(description="List of 3-7 specific, testable criteria an image MUST meet to be relevant.")
    exclusion_criteria: List[str] = Field(description="List of 2-5 criteria that EXCLUDE an image from relevance.")
    adjudication_logic: str = Field(description="A clear IF-THEN-ELSE statement defining PASS/FAIL conditions.")
    verification_steps: List[str] = Field(description="Ordered list of 5-9 concrete, criteria-derived visual verification steps the auditing vision-LLM must execute verbatim on every candidate image (full-canvas region sweep including tiny thumbnails/favicons/watermarks, shape & geometry, typography & exact spelling, color & gradient treatment, person identity when applicable, final adjudication).")
    search_keywords: List[str] = Field(description="List of 5-10 single-word search terms (e.g. ['woman', 'female', 'portrait']) rather than multi-word phrases, to ensure broad keyword match capability in indexed assets.")
    vision_tag_filter: List[str] = Field(description="List of 2-5 keywords specifically mapped to Google Cloud Vision API tag vocabulary.")
    audit_instructions: str = Field(description="Detailed instructions to be passed to the auditing LLM describing the criteria.")
    extraction_schema: Dict[str, str] = Field(description="Dynamic key-value pairs representing additional boolean/integer/string properties to extract from the image to verify the audit criteria.")

def get_image_mime_type(path: str) -> str:
    """Helper to dynamically resolve visual asset MIME type based on file path extension."""
    lower_path = path.lower()
    if lower_path.endswith(".png"):
        return "image/png"
    elif lower_path.endswith(".webp"):
        return "image/webp"
    elif lower_path.endswith(".gif"):
        return "image/gif"
    return "image/jpeg"

def generate_audit_config(user_goal: str, reference_image_description: Optional[str] = None, available_tags: Optional[List[str]] = None, reference_image_path: Optional[str] = None) -> dict:
    """Translates a high-level user goal into a structured audit context with strict visual grounding and
    anti-hallucination guardrails. When reference_image_path is given (and no pre-computed description),
    the forensic reference-image analysis is FUSED into this same single multimodal LLM call — the model
    sees the image directly and writes the forensic breakdown into `image_description` itself."""
    image_context = ""
    forensic_directive = ""
    contents = []

    if reference_image_path and not reference_image_description:
        ref_mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            contents.append(types.Part.from_uri(file_uri=reference_image_path, mime_type=ref_mime))
        else:
            with open(reference_image_path, "rb") as f:
                contents.append(types.Part.from_bytes(data=f.read(), mime_type=ref_mime))
        forensic_directive = """
STEP 0: FORENSIC REFERENCE IMAGE ANALYSIS (perform FIRST; write the result into `image_description`):
The reference image is attached to this request. Perform an exhaustive, forensic-level executive breakdown of it (target 150-250 words) and write it into the `image_description` field, covering:
1. Target Subject & Scope: the exact brand, product, UI component, person, or visual subject shown (do not mix in separate brands/products/subjects).
2. Primary Asset Hierarchy & Category: Standalone Brand Logo vs UI Component (button, modal, banner, badge) vs Composite Canvas (hero banner, screenshot, collage, real-world photograph) vs Person/Product Photograph vs Illustration.
3. Exact Visual Signatures: exact wordmarks, typography, geometric shapes and their construction, border radius, padding, layout structure — and for persons, distinctive identifying features (face, hair, pose, clothing).
4. Color & Contrast Styling: exact colors, gradients, contrast levels, and shadow/elevation effects.
5. Compliance & Style Classification (brand assets only): using YOUR OWN knowledge of that specific brand's official design-system evolution, classify whether the shown style is the brand's CURRENT/ACTIVE standard or an OUTDATED/LEGACY/deprecated iteration, and briefly describe what the other state looks like. If the brand is unknown to you or the subject is not a brand asset, say so and describe only what is visually verifiable — do NOT invent compliance rules.
Ground every criterion, keyword, and instruction below in THIS image analysis. The system is brand-agnostic: never assume a specific brand beyond what the image and the user goal establish.
"""
    elif reference_image_description:
        image_context = f"\nReference Image Description: {reference_image_description}\n"

    # Hard cap: the tag pool is prompt GUIDANCE only, never the full DB vocabulary (token-overflow guard).
    # Self-contained fallback: the optional WEB_AUDIT_VISION_TAGS global is used only if it exists.
    default_tags = globals().get("WEB_AUDIT_VISION_TAGS") or [
        "logo", "icon", "font", "graphics", "screenshot", "banner",
        "illustration", "photograph", "text", "person", "product", "brand"
    ]
    active_tags = list(available_tags if available_tags else default_tags)[:150]
    tags_pool_str = ", ".join([f"'{t}'" for t in active_tags])

    builder_prompt = f"""
You are a Lead Enterprise Visual & UI/UX Asset Auditor building a high-precision audit configuration for executive leadership.
Your task is to expand a user's audit goal into an airtight, zero-mistake structured audit context.

User Goal: {user_goal}
{image_context}{forensic_directive}

BRAND COMPLIANCE SIGNATURES INFERENCE:
Analyze the User Goal and the Reference Image Description (if provided) to extract:
1. The target brand, UI component, person, or visual subject under audit (e.g. a payment brand logo, a video-platform icon, a specific favicon, a cookie banner, a product suite's logos, or a specific person).
2. What constitutes the COMPLIANT (active, modern, approved) visual design, layout, typography, or shape.
3. What constitutes the NON-COMPLIANT (legacy, outdated, spoofed, or incorrect) visual design, layout, typography, or shape.
Ground all inclusion and exclusion criteria strictly in these inferred compliance signatures.

STEP 1: DYNAMIC BRAND & INTENT EXTRACTION
Identify the target brand/visual subject under audit based on the brand compliance signatures inference. Restrict all criteria, search keywords, and instructions strictly to this extracted subject.

STRICT VISUAL GROUNDING & ANTI-HALLUCINATION (CRITICAL):
- You MUST base all inclusion/exclusion criteria and visual descriptions strictly and exclusively on what is physically visible inside the provided reference image.
- Do NOT assume, extrapolate, or hallucinate the presence of brand names, wordmarks, UI elements, button texts, or logos that are cropped out or missing from the reference image.
- If a button, text, or logo is not visible in the reference image, do NOT include it as a mandatory requirement (inclusion criteria).

STEP 2: COMPLIANCE STATE ALIGNMENT & TEMPLATE ROLE ANALYSIS
Analyze the role of the reference image template (if provided) using your inferred compliance signatures.
- **State Conflict (Negation/Comparative Match)**: If the reference image represents the *Active/Compliant/New* standard, but the user wants to find *outdated/old* assets.
  * The template role is **Negative / Comparative Match**.
  * The exclusion criteria MUST exclude the reference image's compliant visual signatures.
  * The inclusion criteria MUST target older legacy styles of that same brand.
- **State Alignment (Positive Template Match)**: If the reference image represents the *Outdated/Legacy/Old* standard, and the user wants to find *outdated/old* assets.
  * The template role is **Positive Template Match**.
  * The inclusion criteria MUST require matching the reference image's legacy visual signatures.
  * The exclusion criteria MUST explicitly exclude the modern, compliant standard.
- **General Discovery Match**: If the user wants to find *all* assets of the brand regardless of state.
  * The inclusion criteria should pass the reference style AND other iterations of the brand.

STEP 2.5: MATCH MODE INFERENCE (match_mode) (CRITICAL):
Infer how strictly candidates must match, from the user's wording, and set `match_mode`:
- 'exact' (e.g. "this exact image"): candidates must contain the SAME image/asset — identical composition and subject. Scale, compression, resolution loss, or appearing as a SMALL EMBEDDED element inside a bigger page/screenshot still count as the same image. Different crops, poses, styles, or variants do NOT.
- 'similar' (e.g. "similar images", "images like this"): candidates count in ANY variation of the target subject — cropped or partial views (e.g. only the mascot's head), blurred, pixelated, low-resolution, recolored, resized, mirrored, or embedded instances. Criteria must be keyed on the subject's core STRUCTURAL SIGNATURES, never on full-image equality.
- 'discovery' (e.g. "all assets of this brand"): every iteration/version of the subject counts, per the state-alignment rules above.
Default to 'similar' when the wording is ambiguous. The `inclusion_criteria`, `exclusion_criteria`, and `adjudication_logic` MUST all be written consistently with the chosen mode — in 'similar' mode they must EXPLICITLY state that cropped, blurred, partial, and small embedded instances of the subject still satisfy the criteria; in 'exact' mode they must EXPLICITLY reject variants while still accepting the exact image at any scale or embedded inside a composite.

CONTEXTUAL RETRIEVAL MANDATE for search_keywords (CRITICAL):
The target often appears as a SMALL element inside larger composites (a profile picture on a login page, a favicon in a header, a badge in a footer). Include 2-3 CONTEXT keywords describing where such an element typically lives (e.g. 'avatar', 'profile', 'icon', 'banner', 'screenshot') alongside the subject keywords, so text search also surfaces composite canvases whose descriptions mention the embedded element.

STEP 3: REFERENCE COMPOSITE CANVAS DETECTION
Evaluate the description of the reference image:
- Set `reference_is_composite_canvas` to True if it describes a composite scene, real-world photograph, or webpage screenshot containing the logo/asset.
- Set `reference_is_composite_canvas` to False only if it represents an isolated, standalone logo on a plain background.

STEP 4: STRICT BRAND EXCLUSION & ANTI-SPOOFING GUARDRAIL (CRITICAL)
If the target visual subject is a specific sub-brand or product logo:
- You MUST explicitly include in the `exclusion_criteria` a rule to exclude the generic corporate master logo unless it is explicitly accompanied by the sub-brand.
- ANTI-SPOOFING: Add explicit rules to reject look-alike misspellings or related but incorrect sub-brands.

STEP 5: COLOR & CANVAS CONTEXT INDEPENDENCE (CRITICAL)
Unless the user's goal explicitly specifies a color constraint, you MUST explicitly write in the `audit_instructions` and `adjudication_logic` that color is not a match determinant.

STEP 6: STEP-BY-STEP VERIFICATION PLAN (verification_steps) (CRITICAL):
Produce an ordered `verification_steps` list (5-9 steps) that the downstream auditing vision-LLM will execute verbatim on every candidate image. Derive each step from the criteria above. The plan MUST include, in this order:
1. FULL-CANVAS REGION SWEEP: inspect every region of the candidate image (all four corners, header, footer, navigation, buttons, background, center) and enumerate EVERY logo/brand/graphic element found — explicitly including tiny thumbnails, favicons, app icons, watermarks, and partially occluded or low-resolution marks. A valid target match ANYWHERE on the canvas counts, no matter how small.
2. SHAPE & GEOMETRY: verify the structural shape/geometry of each detected target element against the compliance signatures (e.g. interlocking loops vs wordmark, border radius, proportions).
3. TYPOGRAPHY & EXACT SPELLING: verify wordmarks letter-by-letter; reject lookalikes and misspellings.
4. COLOR & GRADIENT: verify the gradient/solid color treatment of each detected element, applying the color-independence rule unless the goal explicitly constrains color.
5. PERSON IDENTITY (only if the audit subject involves a person): verify that the SAME person from the reference image appears (facial features, appearance) — no lookalikes.
6. FINAL ADJUDICATION: apply `adjudication_logic` to the collected evidence.

CRITICAL RULE FOR vision_tag_filter:
Propose 2-5 SHORT, generic Google Cloud Vision-style label words for the target subject (e.g. 'logo', 'font', 'screenshot', 'graphics').
Prefer tags from this example vocabulary when applicable: [{tags_pool_str}]
Your proposals are grounded against the full database vocabulary afterwards, so generic single label words outperform brand-specific phrases.

CRITICAL RULE FOR search_keywords (CRITICAL):
- The search_keywords MUST be a list of single-word search terms rather than multi-word phrases.
- RECALL MANDATE: include the brand's short forms, abbreviations, and common filename tokens (e.g. 'gpay' as well as 'pay', 'yt' as well as 'youtube') so keyword search over filenames and descriptions can catch assets whose text descriptions are sparse.

CRITICAL MANDATES FOR OPTICAL RESOLUTION & CLARITY AUDITS (CRITICAL):
1. **Strict Adjudication Logic**: Write a crystal-clear IF-THEN-ELSE statement in `adjudication_logic`.
   - Recognize Cropped/Blurred Targets: State explicitly that if the target visual asset is cropped, low-resolution, or blurred, but you can still identify its core structural signatures, it remains eligible.
   - Flag Resolution Status: In such cases, you must mark `optical_resolution_sufficient = False` in the extraction schema.
   - Only evaluate `matches_criteria = False` if the image is so extremely degraded, pixelated, or tiny (sub-pixel) that it is mathematically impossible to distinguish it from a generic shape.
2. **Diagnostic Extraction Schema**: In `extraction_schema`, define:
   - `detected_asset_style`: string (Exact description of what is seen on canvas)
   - `is_outdated_or_noncompliant`: boolean
   - `is_embedded_in_composite_hero`: boolean
   - `composite_location_notes`: string
   - `optical_resolution_sufficient`: boolean
"""

    contents.append(builder_prompt)

    # Passing the Pydantic model directly to the SDK (single call: forensics + config together)
    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=contents,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=AuditContextModel,
            temperature=0.0
        )
    )
    return json.loads(response.text)

_TAG_VOCAB_CACHE = None

async def get_relevant_tag_vocabulary(sample_percent: float = 2.0, max_tags: int = 5000) -> List[str]:
    """Fetches the most common vision-tag vocabulary from a fast page SAMPLE of visual_assets
    (frequency-ordered, capped) instead of a full-table DISTINCT unnest scan, and caches it for
    the whole session. Read-only; falls back to a row-bounded scan if TABLESAMPLE is unavailable."""
    global _TAG_VOCAB_CACHE
    if _TAG_VOCAB_CACHE is not None:
        return _TAG_VOCAB_CACHE
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            try:
                rows = await db.fetch(
                    f"""
                    SELECT tag FROM (
                        SELECT unnest(vision_tags) AS tag
                        FROM {DB_SCHEMA}.visual_assets TABLESAMPLE SYSTEM ({float(sample_percent):.2f})
                    ) t
                    GROUP BY tag ORDER BY count(*) DESC
                    LIMIT $1
                    """,
                    max_tags
                )
            except Exception:
                rows = await db.fetch(
                    f"""
                    SELECT tag FROM (
                        SELECT unnest(vision_tags) AS tag
                        FROM (SELECT vision_tags FROM {DB_SCHEMA}.visual_assets LIMIT 150000) s
                    ) t
                    GROUP BY tag ORDER BY count(*) DESC
                    LIMIT $1
                    """,
                    max_tags
                )
        _TAG_VOCAB_CACHE = [r["tag"] for r in rows if r["tag"]]
        return _TAG_VOCAB_CACHE
    except Exception as e:
        print(f"Warning: tag vocabulary sampling unavailable ({e}); proposed tags used as-is (will re-probe next run).")
        return []

def ground_tags_against_vocabulary(proposed_tags: List[str], search_keywords: List[str], vocabulary: List[str], max_tags: int = 8) -> List[str]:
    """Maps LLM-proposed tags onto the REAL database vocabulary (closest-spelling substring matches,
    frequency-ordered). Recall-safe: proposals with no vocabulary hit are kept, since the tag search
    arm matches by ILIKE substring anyway."""
    proposals = [p.strip() for p in (proposed_tags or []) if p and p.strip()]
    if not vocabulary:
        return proposals[:max_tags]

    probes, seen = [], set()
    for p in proposals + [k.strip() for k in (search_keywords or []) if k]:
        pl = p.lower()
        if pl and pl not in seen:
            seen.add(pl)
            probes.append(pl)

    grounded, used = [], set()
    for probe in probes:
        hits = [v for v in vocabulary if probe in v.lower() or v.lower() in probe]
        hits.sort(key=lambda v: abs(len(v) - len(probe)))  # closest spelling first, frequency order among ties
        for h in hits[:2]:
            hl = h.lower()
            if hl not in used:
                used.add(hl)
                grounded.append(h)

    for p in proposals:  # keep unmatched proposals too
        if p.lower() not in used:
            used.add(p.lower())
            grounded.append(p)

    return grounded[:max_tags]

def build_fused_query_text(context_dict: dict, is_negative: bool = False) -> str:
    """Combines all context fields into a single rich text representation for embedding."""
    if is_negative:
        return f"Exclude images that: {'; '.join(context_dict.get('exclusion_criteria', []))}. Specifically exclude spoofed misspellings, lookalike brands, and other products."

    parts = [
        f"Audit goal: {context_dict.get('audit_goal')}",
        f"Include images that: {'; '.join(context_dict.get('inclusion_criteria', []))}",
        "IMPORTANT FOR EMBEDDING SIMILARITY: Ignore foreground and background color differences. Focus purely on shape, text layout, logo structural design, and semantic meaning."
    ]
    if context_dict.get("image_description"):
        parts.append(f"Reference image: {context_dict.get('image_description')}")
    return " | ".join(parts)

def _embed_single(contents) -> List[float]:
    """One unambiguous embed_content call -> plain python vector."""
    return client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=768)
    ).embeddings[0].values

def _l2_normalize(vec) -> List[float]:
    v = np.asarray(vec, dtype=np.float32)
    n = np.linalg.norm(v)
    return (v / n).tolist() if n > 0 else v.tolist()

async def embed_query_probes(context_dict: dict, reference_image_path: Optional[str] = None) -> Dict[str, List[float]]:
    """Generates ALL query-side probe vectors concurrently for multi-probe retrieval:
       - 'text'     : positive fused audit text probe
       - 'negative' : exclusion-criteria text probe (contrastive demotion)
       - 'image'    : reference image probe (skipped gracefully if the embed model rejects images)
       - 'fused'    : L2-normalized average of image+text probes (multimodal probe)
    Each probe is embedded in its OWN call so image and text signals never mask each other."""
    pos_text = build_fused_query_text(context_dict, is_negative=False)[:1500]
    neg_text = build_fused_query_text(context_dict, is_negative=True)[:1500]

    loop = asyncio.get_running_loop()
    text_future = loop.run_in_executor(_EMBED_EXECUTOR, _embed_single, pos_text)
    neg_future = loop.run_in_executor(_EMBED_EXECUTOR, _embed_single, neg_text)

    image_future = None
    if reference_image_path:
        mime = get_image_mime_type(reference_image_path)
        if reference_image_path.startswith("gs://"):
            img_part = types.Part.from_uri(file_uri=reference_image_path, mime_type=mime)
        else:
            with open(reference_image_path, "rb") as f:
                img_part = types.Part.from_bytes(data=f.read(), mime_type=mime)
        image_future = loop.run_in_executor(_EMBED_EXECUTOR, _embed_single, [img_part])

    probes = {}
    probes["text"], probes["negative"] = await asyncio.gather(text_future, neg_future)
    if image_future is not None:
        try:
            probes["image"] = list(await image_future)
            probes["fused"] = _l2_normalize(
                np.asarray(_l2_normalize(probes["image"]), dtype=np.float32) + np.asarray(_l2_normalize(probes["text"]), dtype=np.float32)
            )
        except Exception as e:
            print(f"Warning: reference-image embedding probe unavailable ({e}). Falling back to text-only retrieval probes.")
    # L2-normalize every probe: cosine ranking is scale-invariant, and unit probes keep
    # L2/IP-opclass ANN indexes rank-equivalent with convertible distances.
    return {k: _l2_normalize(v) for k, v in probes.items()}

async def embed_audit_context(context_dict: dict, reference_image_path: Optional[str] = None) -> Tuple[List[float], List[float]]:
    """Backwards-compatible wrapper returning a (positive, negative) embedding pair."""
    probes = await embed_query_probes(context_dict, reference_image_path)
    pos_vec = probes.get("fused") or probes.get("image") or probes["text"]
    return pos_vec, probes["negative"]

In [ ]:
# 2. Parallel Hybrid Search with RRF & Drop-Off Detection
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
import json
import time
import asyncio
from pgvector.asyncpg import register_vector
from concurrent.futures import ThreadPoolExecutor
from pydantic import BaseModel, Field

# ---- Retrieval tunables (recall vs latency) ----
VECTOR_ARM_LIMIT = 1000          # full mode: per-probe ANN candidates; MUST be <= hnsw.ef_search (pgvector caps ef_search at 1000)
QUICK_VECTOR_ARM_LIMIT = 200     # quick mode: fast, high-precision ANN head per probe
FULL_TEXT_ARM_CAP = 100000       # full mode: FTS / tag / filename arms may scan up to 1 lakh rows each
QUICK_TEXT_ARM_LIMIT = 1000      # quick mode: bounded text arms
CONTRASTIVE_NEG_WEIGHT = 0.3     # client-side demotion weight for negative-probe similarity
VECTOR_SAFEGUARD_DIST = 0.28     # absolute visual-similarity rescue threshold
CROSS_ENCODER_SKIP_N = 60        # at/below this candidate count, skip text reranking entirely
CROSS_ENCODER_BATCH = 25         # candidates scored per single LLM call

_INDEX_CATALOG_CACHE = None

def _extract_gin_expr(indexdef: str) -> Optional[str]:
    """Extracts the exact indexed expression from a 'USING gin (...)' indexdef (paren-balanced)."""
    dl = indexdef.lower()
    key = "using gin ("
    i = dl.find(key)
    if i < 0:
        return None
    start = i + len(key)
    depth = 1
    for j in range(start, len(indexdef)):
        c = indexdef[j]
        if c == '(':
            depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0:
                return indexdef[start:j].strip()
    return None

async def get_index_catalog() -> dict:
    """One-time (cached) inspection of the indexes on visual_assets, so every search arm can be shaped
    to actually HIT them. An expression GIN index only fires on an EXACT expression match, and a missed
    index means a full multi-million-row scan — this is where retrieval latency comes from."""
    global _INDEX_CATALOG_CACHE
    if _INDEX_CATALOG_CACHE is not None:
        return _INDEX_CATALOG_CACHE

    catalog = {"ann": False, "vector_ops": "cosine", "fts_expr": None, "fts_config": "english",
               "filename_trgm": False, "tags_gin": False, "indexdefs": []}
    try:
        engine, _ = await get_alloydb_connection()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                "SELECT indexdef FROM pg_indexes WHERE schemaname = $1 AND tablename = 'visual_assets'",
                DB_SCHEMA
            )
        import re
        for r in rows:
            d = r["indexdef"]
            catalog["indexdefs"].append(d)
            dl = d.lower()
            if ("using hnsw" in dl or "using ivfflat" in dl or "scann" in dl) and "embedding" in dl:
                catalog["ann"] = True
                m_ops = re.search(r"vector_(cosine|l2|ip)_ops", dl)
                if m_ops:
                    catalog["vector_ops"] = m_ops.group(1)
                elif re.search(r"using scann \(embedding\s+(l2|dot_product)", dl):
                    catalog["vector_ops"] = {"l2": "l2", "dot_product": "ip"}[re.search(r"using scann \(embedding\s+(l2|dot_product)", dl).group(1)]
            if "gin_trgm_ops" in dl and "asset_filename" in dl:
                catalog["filename_trgm"] = True
            expr = _extract_gin_expr(d)
            if expr:
                el = expr.lower()
                if "to_tsvector" in el and "gemini_description" in el:
                    catalog["fts_expr"] = expr
                    m = re.search(r"'(\w+)'::regconfig", expr)
                    if m:
                        catalog["fts_config"] = m.group(1)
                elif el.startswith("vision_tags"):
                    catalog["tags_gin"] = True
        ann_label = f"YES[{catalog['vector_ops']}]" if catalog["ann"] else "no"
        print("[Index Catalog] ANN(embedding)=%s | GIN tsvector(description)=%s | GIN(vision_tags)=%s | trgm(asset_filename)=%s"
              % (ann_label, *("YES" if v else "no" for v in (bool(catalog["fts_expr"]), catalog["tags_gin"], catalog["filename_trgm"]))))
        if catalog["fts_expr"]:
            print(f"[Index Catalog] FTS arm bound to indexed expression: {catalog['fts_expr']} (config: {catalog['fts_config']})")
        else:
            print("[Index Catalog] No GIN tsvector index detected on gemini_description — FTS arm uses the default expression (may seq-scan).")
        _INDEX_CATALOG_CACHE = catalog  # cache only successful inspections
    except Exception as e:
        print(f"[Index Catalog] inspection unavailable ({e}); using default arm expressions this run (will re-probe next search).")
    return catalog

class _BatchScoreItem(BaseModel):
    index: int = Field(description="The candidate index exactly as given in the input list.")
    score: int = Field(description="Relevance score 0-100 for that candidate.")

class _BatchScores(BaseModel):
    scores: List[_BatchScoreItem] = Field(description="One score entry for EVERY candidate index in the input.")

async def run_semantic_reranking_and_filter(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_context: dict, max_workers: int = 25) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies LLM Cross-Encoder semantic scoring in BATCHED calls (~25 candidates per call instead of
    1 call per candidate), with visual-vector and keyword-promotion safeguards to protect recall.
    Small candidate sets skip this stage entirely — the vision inference stage is the real judge."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")

    # Fast path: for small sets a text-only filter can only add latency and lose recall.
    if len(candidates) <= CROSS_ENCODER_SKIP_N:
        print(f"[Cross-Encoder] {len(candidates)} candidates <= {CROSS_ENCODER_SKIP_N}: skipping text reranking (vision inference will adjudicate all of them).")
        return df_high.reset_index(drop=True), df_edge.reset_index(drop=True), pd.DataFrame()

    audit_goal = audit_context.get("audit_goal", "")
    inclusion_str = "; ".join(audit_context.get("inclusion_criteria", []))

    # 1. Visual Vector Safeguard (absolute visual distance check) — bypasses text scoring entirely.
    prescored, to_score = [], []
    for row in candidates:
        if row.get("vector_distance", 1.0) < VECTOR_SAFEGUARD_DIST:
            prescored.append({
                **row,
                "cross_encoder_score": 100,  # Max score to force-promote
                "relevance_score": row.get("relevance_score", 0.0) * 2.0,  # Visual match boost
                "vector_safeguard_triggered": True
            })
        else:
            to_score.append(row)

    def _score_batch(batch: list) -> list:
        lines = []
        for j, row in enumerate(batch):
            desc = str(row.get("gemini_description") or "")[:400]
            tags = ", ".join([str(t) for t in (row.get("vision_tags") or [])])[:200]
            fname = str(row.get("asset_filename") or "")[:120]
            lines.append(f"[{j}] filename: {fname} | tags: {tags} | description: {desc}")
        joined = "\n".join(lines)

        prompt = f"""You are a rapid relevance scoring engine. Score EVERY candidate below for relevance to the audit goal.
Audit Goal: {audit_goal}
Inclusion Criteria: {inclusion_str}

Score each candidate's textual relevance 0-100 using this calibrated rubric:
- 80-100 (High): Direct matches to the target subject, clear presence of target visual elements, or standalone target brand logos.
- 30-79 (Borderline): Contextual matches, related terms/brands, composite graphics/screenshots/hero banners that may CONTAIN the target somewhere (even small), or descriptions with visual layout complexity.
- 0-29 (Low): Clearly unrelated subjects (different products, unrelated graphics).

IMPORTANT RECALL RULE: A sparse, generic, or missing description is NOT evidence of irrelevance — the target may appear as a small element the description skipped. Score such candidates as Borderline (30-79), never Low.

Candidates:
{joined}

Return one score entry for every index 0..{len(batch) - 1}."""
        try:
            resp = client.models.generate_content(
                model=GEMINI_CROSS_ENCODER_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=_BatchScores,
                    temperature=0.0
                )
            )
            parsed = {int(s["index"]): int(s["score"]) for s in json.loads(resp.text)["scores"]}
        except Exception as e:
            print(f"[Cross-Encoder] Batch scoring failed ({e}); defaulting batch to neutral 50.")
            parsed = {}
        return [parsed.get(j, 50) for j in range(len(batch))]

    batches = [to_score[i:i + CROSS_ENCODER_BATCH] for i in range(0, len(to_score), CROSS_ENCODER_BATCH)]
    print(f"[Cross-Encoder] Semantic reranking of {len(to_score)} candidates in {len(batches)} batched LLM calls (+{len(prescored)} vector-safeguard promotions)...")

    loop = asyncio.get_running_loop()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        batch_scores = await asyncio.gather(*[loop.run_in_executor(executor, _score_batch, b) for b in batches])

    reranked = list(prescored)
    for batch, scores in zip(batches, batch_scores):
        for row, score in zip(batch, scores):
            ce_mult = max(0.1, score / 50.0)
            reranked.append({
                **row,
                "cross_encoder_score": score,
                "relevance_score": row.get("relevance_score", 0.0) * ce_mult,
                "vector_safeguard_triggered": False
            })

    high_list, edge_list, low_list = [], [], []
    for item in reranked:
        if item.get("vector_safeguard_triggered", False):
            filename = item.get("asset_filename") or str(item.get("gcs_raw_path", "")).split("/")[-1]
            dist = item.get("vector_distance", 0)
            print(f"🛡️ Vector Safeguard triggered: Force-promoted {str(filename)[:40]} due to high visual similarity (distance: {dist:.4f})")
            high_list.append(item)
            continue

        score = item["cross_encoder_score"]
        if score >= 75:
            high_list.append(item)
        elif score >= 30:
            edge_list.append(item)
        elif item.get("promoted_by_keyword_or_tag", False):
            # Recall safeguard: exact keyword/tag hits are never text-filtered out of the audit.
            item["cross_encoder_score"] = 30
            edge_list.append(item)
        else:
            low_list.append(item)

    df_high_final = pd.DataFrame(high_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if high_list else pd.DataFrame()
    df_edge_final = pd.DataFrame(edge_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if edge_list else pd.DataFrame()
    df_low_final = pd.DataFrame(low_list).sort_values(by="relevance_score", ascending=False).reset_index(drop=True) if low_list else pd.DataFrame()

    return df_high_final, df_edge_final, df_low_final

def compute_weighted_rrf_rerank(candidates: list, audit_context: dict) -> list:
    """Zero-latency in-memory multi-factor reranker. Executes in local CPU RAM (< 0.5ms) without external API overhead."""
    tag_filter = [t.lower().strip() for t in audit_context.get("vision_tag_filter", []) if t]
    search_keywords = [k.lower().strip() for k in audit_context.get("search_keywords", []) if k]
    reranked = []

    for item in candidates:
        row = item["data"]
        base_score = item["score"]
        multiplier = 1.0

        # 1. Vision tag exact hit boost (2.0x per matching tag up to 8x)
        tags = [str(t).lower() for t in (row.get("vision_tags") or [])]
        tag_hits = sum(1 for t in tags if any(ft in t or t in ft for ft in tag_filter))
        if tag_hits > 0:
            multiplier *= (2.0 ** min(tag_hits, 3))

        # 2. Keyword exact hit in filename or description boost (1.5x per matching keyword up to 2.25x)
        desc = str(row.get("gemini_description") or "").lower()
        fname = str(row.get("asset_filename") or "").lower()
        kw_hits = sum(1 for kw in search_keywords if kw in desc or kw in fname)
        if kw_hits > 0:
            multiplier *= min(1.5 ** kw_hits, 2.25)

        reranked.append({
            "data": row,
            "score": base_score * multiplier,
            "base_rrf_score": base_score,
            "rerank_multiplier": multiplier,
            "vector_distance": item.get("vector_distance", 1.0)
        })

    reranked.sort(key=lambda x: x["score"], reverse=True)
    return reranked

async def run_hybrid_search(scope_config: dict, audit_context: dict, reference_image_path: Optional[str] = None, limit: int = 10000, quick_mode: bool = False) -> List[dict]:
    """Performs parallel MULTI-PROBE Vector + Keyword/Filename + Tag search with RRF fusion, local reranking,
    and deduplication (No silent cliff).

    Modes: quick_mode=True keeps every arm small and fast (top-200 ANN head per probe with a matching
    ef_search, text arms capped at 1000) — the best candidates within a tight latency budget.
    quick_mode=False (Full Scan) uses the deep ANN head (1000/probe) and lets the text arms scan up to
    `limit` rows each (1 lakh when called by the Full Scan pipeline).

    Latency: every arm is shaped to HIT the physical indexes discovered via get_index_catalog() —
    vector arms order by a bare `embedding <=> $probe` (HNSW/ScaNN ANN scan; contrastive negative
    demotion applied client-side), the FTS arm uses the EXACT indexed tsvector expression (GIN bitmap
    scan — no coalesce()/OR wrappers that would force a seq scan), the tag arm uses `vision_tags && $tags`
    (GIN array-overlap operator; tags are DB-grounded upstream so exact match is correct), and the
    filename ILIKE arm only runs when a pg_trgm index exists for it.
    Recall: up to three vector probes (fused image+text, image-only, text-only) give diluted composite
    embeddings (tiny thumbnails inside screenshots) multiple chances to surface."""
    probes, catalog = await asyncio.gather(
        embed_query_probes(audit_context, reference_image_path),
        get_index_catalog()
    )

    engine, _ = await get_alloydb_connection()

    search_keywords = [k for k in audit_context.get("search_keywords", []) if k]
    keyword_query_str = " OR ".join(search_keywords) if search_keywords else ""
    fname_patterns = [f"%{k.lower().strip()}%" for k in search_keywords]
    exact_tags = [t for t in audit_context.get("vision_tag_filter", []) if t]

    vec_limit = min(limit, QUICK_VECTOR_ARM_LIMIT if quick_mode else VECTOR_ARM_LIMIT)
    text_limit = min(limit, QUICK_TEXT_ARM_LIMIT if quick_mode else FULL_TEXT_ARM_CAP)

    # left() keeps the wire payload small (~12k rows retrieved); downstream stages never need more.
    SELECT_COLS = "asset_id, gcs_raw_path, format, vision_tags, left(gemini_description, 600) AS gemini_description, asset_filename, page_url, content_hash"

    fts_expr = catalog.get("fts_expr") or "to_tsvector('english', gemini_description)"
    fts_config = catalog.get("fts_config") or "english"

    # Use the distance operator the ANN index actually serves — a mismatched operator (e.g. cosine
    # <=> against a vector_l2_ops index) silently degrades to a full-table distance scan.
    vec_op = {"cosine": "<=>", "l2": "<->", "ip": "<#>"}.get(catalog.get("vector_ops", "cosine"), "<=>")
    if vec_op != "<=>":
        print(f"[Retrieval] ANN opclass is '{catalog['vector_ops']}' — querying with {vec_op} so the index is served; distances converted to cosine scale downstream.")

    async def _tune_ann_recall(db):
        # ef_search must be >= the vector arm LIMIT or HNSW silently returns fewer rows (pgvector cap: 1000).
        for guc in (f"SET hnsw.ef_search = {min(max(vec_limit, 100), 1000)}", "SET ivfflat.probes = 20", "SET scann.num_leaves_to_search = 200"):
            try:
                await db.execute(guc)
            except Exception:
                pass

    # Run the vector probe arms + keyword arm + tag arm + filename arm concurrently on pooled connections.
    def _to_cosine_scale(d):
        # Unit-normalized probes: L2 d^2/2 and 1+inner_product approximate the cosine distance.
        if d is None:
            return None
        d = float(d)
        if vec_op == "<->":
            return min(max((d * d) / 2.0, 0.0), 2.0)
        if vec_op == "<#>":
            return min(max(1.0 + d, 0.0), 2.0)
        return d

    async def run_vector(arm_name: str, probe_vec, explain_plan: bool = False):
        t_arm = time.time()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            await register_vector(db)
            await _tune_ann_recall(db)
            if explain_plan:
                # Ground truth on index usage, printed BEFORE the fetch so a slow scan is diagnosed live.
                try:
                    plan = await db.fetch(
                        f"EXPLAIN SELECT asset_id FROM {DB_SCHEMA}.visual_assets ORDER BY embedding {vec_op} $1::vector LIMIT 10",
                        probe_vec
                    )
                    plan_text = " | ".join(p["QUERY PLAN"] for p in plan[:3])
                    if "Index Scan" in plan_text or "index" in plan_text.lower():
                        print(f"[Index Catalog] Vector plan check: ANN index IS used -> {plan_text[:140]}")
                    else:
                        print(f"[Index Catalog] ⚠️ WARNING: vector query does NOT use the ANN index (full seq scan!). Plan: {plan_text[:180]}")
                except Exception:
                    pass
            rows = await db.fetch(
                f"""
                SELECT {SELECT_COLS},
                       (embedding {vec_op} $1::vector) AS vector_distance,
                       (embedding {vec_op} $2::vector) AS neg_distance
                FROM {DB_SCHEMA}.visual_assets
                ORDER BY embedding {vec_op} $1::vector
                LIMIT $3
                """,
                probe_vec, probes["negative"], vec_limit
            )
            out = []
            for r in rows:
                if r["vector_distance"] is None:
                    continue
                rd = dict(r)
                rd["vector_distance"] = _to_cosine_scale(rd["vector_distance"])
                rd["neg_distance"] = _to_cosine_scale(rd["neg_distance"])
                out.append(rd)
            # Contrastive demotion (pos_dist - w * neg_dist) computed in-memory, keeping the ANN index usable.
            out.sort(key=lambda r: float(r["vector_distance"]) - CONTRASTIVE_NEG_WEIGHT * float(r["neg_distance"]))
            print(f"[Retrieval] {arm_name} arm done: {len(out)} rows in {time.time() - t_arm:.2f}s")
            return arm_name, out, time.time() - t_arm

    async def run_fts():
        if not keyword_query_str:
            return "fts", [], 0.0
        t_arm = time.time()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            # Two-tier plan: the INNER query stops at the first text_limit GIN matches (no ORDER BY =
            # no rank computation over the entire match set, which can be millions of rows); the rank
            # is evaluated only for the emitted rows and the OUTER query sorts that bounded set.
            rows = await db.fetch(
                f"""
                SELECT * FROM (
                    SELECT {SELECT_COLS},
                           ts_rank_cd({fts_expr}, websearch_to_tsquery('{fts_config}', $1)) AS fts_rank
                    FROM {DB_SCHEMA}.visual_assets
                    WHERE {fts_expr} @@ websearch_to_tsquery('{fts_config}', $1)
                    LIMIT $2
                ) bounded_matches
                ORDER BY fts_rank DESC
                """,
                keyword_query_str, text_limit
            )
            out = [dict(r) for r in rows]
            print(f"[Retrieval] fts arm done: {len(out)} rows in {time.time() - t_arm:.2f}s")
            return "fts", out, time.time() - t_arm

    async def run_tags():
        if not exact_tags:
            return "tags", [], 0.0
        t_arm = time.time()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            # Case-variant expansion keeps the arm on the GIN-indexable exact && operator
            # (covers lower/UPPER/Capitalized storage without any substring scan).
            tag_variants = []
            for t in exact_tags:
                for v in (t, t.lower(), t.upper(), t.capitalize(), t.title()):
                    if v not in tag_variants:
                        tag_variants.append(v)
            rows = await db.fetch(
                f"""
                SELECT {SELECT_COLS}
                FROM {DB_SCHEMA}.visual_assets
                WHERE vision_tags && $1::text[]
                LIMIT $2
                """,
                tag_variants, text_limit
            )
            if not rows and not quick_mode:
                # Full-mode-only recall fallback: substring matching is a full-table scan, so quick
                # mode never pays for it — vector + FTS arms carry recall there instead.
                print("[Retrieval] Tag arm exact-overlap returned 0 rows; retrying with substring ILIKE fallback (Full Scan only)...")
                tag_patterns = [f"%{t.lower().strip()}%" for t in exact_tags]
                rows = await db.fetch(
                    f"""
                    SELECT {SELECT_COLS}
                    FROM {DB_SCHEMA}.visual_assets
                    WHERE EXISTS (
                        SELECT 1 FROM unnest(vision_tags) tag
                        WHERE tag ILIKE ANY($1::text[])
                    )
                    LIMIT $2
                    """,
                    tag_patterns, text_limit
                )
            elif not rows:
                print("[Retrieval] Tag arm: no exact overlap; substring fallback skipped in Quick mode (latency guard).")
            out = [dict(r) for r in rows]
            print(f"[Retrieval] tags arm done: {len(out)} rows in {time.time() - t_arm:.2f}s")
            return "tags", out, time.time() - t_arm

    async def run_filename():
        # Substring ILIKE without a trigram index = guaranteed full-table scan; only run when indexed.
        if not fname_patterns or not catalog.get("filename_trgm"):
            return "filename", [], 0.0
        t_arm = time.time()
        async with engine.connect() as conn:
            raw_conn = await conn.get_raw_connection()
            db = raw_conn.driver_connection
            rows = await db.fetch(
                f"""
                SELECT {SELECT_COLS}
                FROM {DB_SCHEMA}.visual_assets
                WHERE asset_filename ILIKE ANY($1::text[])
                LIMIT $2
                """,
                fname_patterns, text_limit
            )
            out = [dict(r) for r in rows]
            print(f"[Retrieval] filename arm done: {len(out)} rows in {time.time() - t_arm:.2f}s")
            return "filename", out, time.time() - t_arm

    vector_arms = []
    if "fused" in probes:
        vector_arms.append(("vec_fused", probes["fused"]))
    if "image" in probes:
        vector_arms.append(("vec_image", probes["image"]))
    vector_arms.append(("vec_text", probes["text"]))

    if not catalog.get("ann"):
        # No ANN index detected: every vector probe is a full distance scan over the whole table.
        if quick_mode and len(vector_arms) > 1:
            vector_arms = vector_arms[:1]
        print(f"[Retrieval] WARNING: no ANN index detected on `embedding` — vector search will full-scan ({len(vector_arms)} probe(s)). Expect minutes, not seconds, until an HNSW/ScaNN index is confirmed.")

    arm_tasks = [run_vector(name, vec, explain_plan=(i == 0)) for i, (name, vec) in enumerate(vector_arms)] + [run_fts(), run_tags(), run_filename()]
    arm_outputs = await asyncio.gather(*arm_tasks)
    arm_results = {name: rows for name, rows, _ in arm_outputs}
    print("[Retrieval] " + " | ".join(f"{name}: {len(rows)} rows in {dt:.2f}s" for name, rows, dt in arm_outputs))

    fts_results = arm_results.get("fts", [])
    tag_results = arm_results.get("tags", [])
    filename_results = arm_results.get("filename", [])

    promoted_ids = set()
    for r in fts_results[:200]:
        promoted_ids.add(str(r["asset_id"]))
    for r in tag_results[:200]:
        promoted_ids.add(str(r["asset_id"]))
    for r in filename_results[:200]:
        promoted_ids.add(str(r["asset_id"]))

    # Best (minimum) vector distance per asset across all probe arms.
    vector_distance_map = {}
    for name, _ in vector_arms:
        for r in arm_results[name]:
            aid = str(r["asset_id"])
            d = float(r["vector_distance"])
            if aid not in vector_distance_map or d < vector_distance_map[aid]:
                vector_distance_map[aid] = d

    # Multi-Arm RRF Fusion (k=60). Total vector weight 0.60 split evenly across active probes.
    k = 60
    results_map = {}
    per_vec_weight = 0.60 / len(vector_arms)

    def upsert_ranks(results_list, weight=1.0):
        for rank, row in enumerate(results_list):
            img_id = str(row["asset_id"])
            if img_id not in results_map:
                clean = {c: v for c, v in row.items() if c not in ("vector_distance", "neg_distance", "fts_rank")}
                results_map[img_id] = {"data": clean, "score": 0.0, "vector_distance": vector_distance_map.get(img_id, 1.0)}
            results_map[img_id]["score"] += weight / (k + rank + 1)

    for name, _ in vector_arms:
        upsert_ranks(arm_results[name], weight=per_vec_weight)
    if fts_results:
        upsert_ranks(fts_results, weight=0.25)
    if tag_results:
        upsert_ranks(tag_results, weight=0.15)
    if filename_results:
        upsert_ranks(filename_results, weight=0.10)

    fused = list(results_map.values())

    # Zero-Latency In-Memory Reranking (Provides a smooth score curve for Kneedle)
    reranked_fused = compute_weighted_rrf_rerank(fused, audit_context)

    # Deduplication
    seen_identifiers = set()
    deduplicated = []
    for item in reranked_fused:
        row = item["data"]
        img_id = str(row["asset_id"])
        img_identifier = row.get("content_hash") or row.get("gcs_raw_path")
        if img_identifier not in seen_identifiers:
            seen_identifiers.add(img_identifier)
            is_promoted = img_id in promoted_ids
            deduplicated.append({
                **row,
                "relevance_score": item["score"],
                "base_rrf_score": item.get("base_rrf_score", 0),
                "rerank_multiplier": item.get("rerank_multiplier", 1),
                "promoted_by_keyword_or_tag": is_promoted,
                "vector_distance": item.get("vector_distance", 1.0)
            })

    return deduplicated[:limit]

def detect_dropoff_flawless(df: pd.DataFrame, sensitivity: float = 1.0) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Applies Kneedle curvature + rolling volatility to segment candidates into High, Edge, and Low."""
    if df is None or df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    df_sorted = df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)

    # Small-set guard: with <= 25 candidates there is no meaningful curve to segment, and the old
    # equal-scores split could strand a lone candidate in Low. Send them all to High — the
    # cross-encoder skip threshold covers sets this small and vision inference adjudicates.
    if len(df_sorted) <= 25:
        return df_sorted, pd.DataFrame(), pd.DataFrame()

    y = df_sorted["relevance_score"].values
    x = np.arange(len(y))

    y_min, y_max = y.min(), y.max()
    if y_max == y_min:
        # All scores identical -> no ranking signal to segment on; discard nothing.
        return df_sorted, pd.DataFrame(), pd.DataFrame()

    y_norm = (y - y_min) / (y_max - y_min + 1e-9)
    x_norm = x / (len(x) - 1)

    coords = np.column_stack((x_norm, y_norm))
    line_start, line_end = coords[0], coords[-1]
    line_vec = line_end - line_start
    line_vec_norm = line_vec / np.sqrt(np.sum(line_vec**2))
    vec_from_start = coords - line_start
    scalar_proj = np.dot(vec_from_start, line_vec_norm)
    proj_on_line = np.outer(scalar_proj, line_vec_norm)
    dist_to_line = np.sqrt(np.sum((vec_from_start - proj_on_line)**2, axis=1))

    idx1 = np.argmax(dist_to_line)

    window = max(3, int(len(y) * 0.05))
    rolling_std = pd.Series(y_norm).rolling(window=window, center=True).std().fillna(0).values
    noise_threshold = np.mean(rolling_std) * (0.6 / sensitivity)

    idx2 = len(y) - 1
    for i in range(idx1 + 2, len(rolling_std)):
        if rolling_std[i] < noise_threshold:
            idx2 = i
            break

    min_borderline_width = max(10, int((len(y) - idx1) * 0.25))
    if (idx2 - idx1) < min_borderline_width:
        idx2 = min(len(y) - 1, idx1 + min_borderline_width)

    idx1 = max(10, idx1)

    high_df = df_sorted.iloc[:idx1 + 1].copy()
    edge_df = df_sorted.iloc[idx1 + 1: idx2 + 1].copy()
    low_df = df_sorted.iloc[idx2 + 1:].copy()

    # Visual Vector Safeguard: Check if any candidate has extremely high similarity (distance < 0.28)
    # even if it is currently classified in Low_df (or has been discarded).
    # Force rescue these to protect visual recall.
    if "vector_distance" in low_df.columns:
        rescued_vec = low_df[low_df["vector_distance"] < VECTOR_SAFEGUARD_DIST].copy()
        if not rescued_vec.empty:
            edge_df = pd.concat([edge_df, rescued_vec], ignore_index=True)
            low_df = low_df[low_df["vector_distance"] >= VECTOR_SAFEGUARD_DIST].copy()
            print(f"🛡️ Vector Safeguard triggered in Kneedle: Force-rescued {len(rescued_vec)} candidate(s) from Low to Borderline based on high visual similarity.")

    if "promoted_by_keyword_or_tag" in low_df.columns:
        rescued = low_df[low_df["promoted_by_keyword_or_tag"] == True].copy()
        if not rescued.empty:
            edge_df = pd.concat([edge_df, rescued], ignore_index=True)
            low_df = low_df[low_df["promoted_by_keyword_or_tag"] != True].copy()
            print(f"🛡️ Safeguard triggered: Promoted {len(rescued)} composite/diluted candidates from Low to Borderline tier based on exact keyword/tag match.")

    return high_df, edge_df, low_df

## Retrieval, Reranking, and Drop-off Segmentation Engine

This cell defines the core search engine:

- `run_hybrid_search`: Combines Vector, Full-Text, and Tag search arms into a fused RRF list, applying zero-latency keyword boosts.
- `detect_dropoff_flawless`: Curvature-based Kneedle algorithm that truncates irrelevant tail results.

In [ ]:
# 3. Hydration & Parallel LLM Audit Inference
import asyncio
from concurrent.futures import ThreadPoolExecutor
import json
import pandas as pd
from typing import List, Tuple, Optional
from google.genai import types
from google.cloud import storage
from pydantic import create_model, Field
import time
from PIL import Image
import io

# One cached GCS client — creating a client per download costs an auth/channel round-trip every time.
_GCS_CLIENT = None
def _get_storage_client():
    global _GCS_CLIENT
    if _GCS_CLIENT is None:
        _GCS_CLIENT = storage.Client(project=PROJECT_ID)
    return _GCS_CLIENT

# Only truly extreme images are downscaled, so tiny embedded logos/thumbnails stay legible to the model.
MAX_IMAGE_DIM = 3072

# Transient API errors worth retrying (rate limits / server hiccups); everything else fails fast.
_TRANSIENT_ERROR_MARKERS = ("429", "500", "503", "resource_exhausted", "unavailable", "deadline", "internal", "overloaded", "timeout", "timed out")

def _generate_with_retry(make_call, max_retries: int = 3):
    """Retries a Gemini call on transient errors with exponential backoff (1.5s, 3s)."""
    for attempt in range(max_retries):
        try:
            return make_call()
        except Exception as e:
            transient = any(m in str(e).lower() for m in _TRANSIENT_ERROR_MARKERS)
            if attempt == max_retries - 1 or not transient:
                raise
            time.sleep(1.5 * (2 ** attempt))

def process_transparency(image_bytes: bytes, default_bg: Tuple[int, int, int] = (30, 30, 30)) -> bytes:
    """Detects alpha transparency and composites it onto a solid dark background so light elements/text
    remain visible; also caps extreme resolutions (> MAX_IMAGE_DIM px) to cut upload/inference latency."""
    try:
        img = Image.open(io.BytesIO(image_bytes))
        needs_flatten = img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info)
        needs_resize = max(img.size) > MAX_IMAGE_DIM
        if not needs_flatten and not needs_resize:
            return image_bytes
        if needs_flatten:
            img = img.convert('RGBA')
            bg = Image.new("RGBA", img.size, default_bg + (255,))
            img = Image.alpha_composite(bg, img).convert("RGB")
        else:
            img = img.convert("RGB")
        if needs_resize:
            scale = MAX_IMAGE_DIM / float(max(img.size))
            img = img.resize((max(1, int(img.width * scale)), max(1, int(img.height * scale))), Image.LANCZOS)
        out_bytes = io.BytesIO()
        img.save(out_bytes, format="PNG")
        return out_bytes.getvalue()
    except Exception as e:
        print(f"Warning: Failed to preprocess image transparency: {e}")
    return image_bytes

def _prep_image(image_bytes: bytes) -> Tuple[bytes, str]:
    """Returns (bytes, mime) after transparency flattening / extreme-size capping, sniffing the real format."""
    processed = process_transparency(image_bytes)
    if processed is not image_bytes:
        return processed, "image/png"
    try:
        fmt = (Image.open(io.BytesIO(image_bytes)).format or "PNG").lower()
        mime = {"jpeg": "image/jpeg", "png": "image/png", "webp": "image/webp", "gif": "image/gif"}.get(fmt, "image/png")
    except Exception:
        mime = "image/png"
    return image_bytes, mime

def _download_asset_bytes(path: str) -> bytes:
    """Downloads raw bytes from GCS (cached client) or the local filesystem."""
    if path.startswith("gs://"):
        bucket_name = path.split("/")[2]
        blob_name = "/".join(path.split("/")[3:])
        return _get_storage_client().bucket(bucket_name).blob(blob_name).download_as_bytes()
    with open(path, "rb") as f:
        return f.read()

def enforce_zero_false_positives_rules(df: pd.DataFrame) -> pd.DataFrame:
    """Enforces zero-false-positive criteria. Demotes matches if confidence is below 75%."""
    if df.empty:
        return df

    def guardrail_check(row):
        matches = bool(row.get("matches_criteria", False))
        try:
            conf = int(float(row.get("match_confidence", 100)))
        except (TypeError, ValueError):
            conf = 100
        rationale = str(row.get("visual_analysis_step_by_step", "")) + " " + str(row.get("match_rationale", ""))
        if matches and conf < 75:
            row["matches_criteria"] = False
            row["match_rationale"] = f"[GUARDRAIL DEMOTION: Conf {conf}% < 75%] {rationale}"
        return row

    return df.apply(guardrail_check, axis=1)

def build_dynamic_audit_model(extraction_schema: dict):
    """Builds the structured-output Pydantic model ONCE per run (identical for every candidate)."""
    fields = {
        "visual_analysis_step_by_step": (
            str,
            Field(description="CHAIN OF THOUGHT: Execute the numbered VERIFICATION STEPS in order and record dense, factual findings per step BEFORE any conclusion (target under 180 words): every canvas region inspected, EVERY brand/logo/graphic element found (including tiny thumbnails, favicons, watermarks), and the shape / typography / color-gradient findings for each.")
        ),
        "criteria_verdicts": (
            str,
            Field(description="One line per criterion, in order: 'I1: PASS|FAIL — short evidence' for each inclusion criterion, then 'E1: TRIGGERED|CLEAR — short evidence' for each exclusion criterion.")
        ),
        "matches_criteria": (
            bool,
            Field(description="Strict final evaluation: True ONLY if the asset matches the target criteria and passes adjudication_logic (even inside a composite hero banner). False otherwise.")
        ),
        "match_confidence": (
            int,
            Field(description="Match Confidence percentage (0-100%). You must assign < 75 if there is any doubt or visual occlusion.")
        ),
        "match_rationale": (
            str,
            Field(description="A concise final executive rationale explaining exactly why matches_criteria evaluated to True or False based on the visual_analysis_step_by_step.")
        )
    }

    for field_name, field_info in (extraction_schema or {}).items():
        reserved = {"matches_criteria", "match_confidence", "match_rationale", "visual_analysis_step_by_step", "criteria_verdicts",
                    "asset_id", "gcs_raw_path", "page_url", "asset_filename", "format", "vision_tags", "gemini_description",
                    "content_hash", "relevance_score", "vector_distance", "base_rrf_score", "rerank_multiplier",
                    "promoted_by_keyword_or_tag", "cross_encoder_score", "error"}
        if field_name in reserved:
            continue

        t = str
        desc = f"Extracted value for {field_name}"

        if isinstance(field_info, dict):
            ftype = field_info.get("field_type", "string").lower()
            desc = field_info.get("description", desc)
        else:
            ftype = str(field_info).lower()

        if ftype == "boolean":
            t = bool
        elif ftype == "integer":
            t = int
        elif ftype == "number":
            t = float

        fields[field_name] = (t, Field(description=desc))

    return create_model("DynamicAuditModel", **fields)

_DEFAULT_VERIFICATION_STEPS = [
    "FULL-CANVAS REGION SWEEP: Systematically inspect every region of the candidate image (all four corners, header, footer, navigation, buttons, background, center) and enumerate EVERY logo, brand mark, icon, or graphic element found — explicitly including tiny thumbnails, favicons, app badges, watermarks, and partially occluded or low-resolution marks.",
    "SHAPE & GEOMETRY: For each detected element, verify its structural shape and geometry against the target's compliance signatures.",
    "TYPOGRAPHY & EXACT SPELLING: Verify any wordmarks letter-by-letter; reject lookalike or misspelled branding.",
    "COLOR & GRADIENT: Note the color/gradient treatment of each detected element; apply the Color Independence Rule unless the criteria explicitly constrain color.",
    "PERSON IDENTITY (if applicable): If the audit subject involves a specific person, verify the SAME person appears (facial features, appearance) — no lookalikes.",
    "FINAL ADJUDICATION: Apply the Strict Adjudication Rule to the evidence collected above."
]

def build_audit_prompt(audit_config: dict, has_reference_image: bool) -> str:
    """Builds the (identical for every candidate) audit prompt ONCE per run, wiring the config's
    step-by-step verification plan and the reference-image comparison directives into the LLM call."""
    audit_instructions = audit_config.get("audit_instructions", "")
    inclusion_criteria = audit_config.get("inclusion_criteria", [])
    exclusion_criteria = audit_config.get("exclusion_criteria", [])
    adjudication_logic = audit_config.get("adjudication_logic", "")
    is_composite = bool(audit_config.get("reference_is_composite_canvas", False))

    inclusion_str = "\n".join([f"- I{i+1}: {c}" for i, c in enumerate(inclusion_criteria)]) if inclusion_criteria else "- None specified"
    exclusion_str = "\n".join([f"- E{i+1}: {c}" for i, c in enumerate(exclusion_criteria)]) if exclusion_criteria else "- None specified"

    verification_steps = audit_config.get("verification_steps") or _DEFAULT_VERIFICATION_STEPS
    steps_str = "\n".join([f"{i+1}. {s}" for i, s in enumerate(verification_steps)])

    match_mode = str(audit_config.get("match_mode", "similar")).lower()
    if "exact" in match_mode:
        match_mode_rule = """EXACT MATCH MODE: The candidate PASSES only if it contains the SAME image/asset as the target — identical composition and subject. Different scale, compression, resolution loss, or the target appearing as a SMALL EMBEDDED element inside a larger page/banner/screenshot still count as the same image. Different crops, poses, styles, redesigns, or variants of the subject MUST FAIL."""
    elif "discovery" in match_mode:
        match_mode_rule = """DISCOVERY MODE: The candidate PASSES if it contains ANY iteration or version of the target subject/brand that satisfies the criteria, regardless of design state, style era, or rendition quality."""
    else:
        match_mode_rule = """SIMILAR MATCH MODE: The candidate PASSES if the target subject is recognizably present in ANY variation — cropped or partial views (e.g. only the mascot's head), blurred, pixelated, low-resolution, recolored, resized, or embedded as a small element inside a composite. Judge by the subject's core structural signatures, NOT by full-image equality; only reject when the depiction is so degraded that identification is impossible or the criteria are clearly violated."""

    reference_instructions = ""
    if has_reference_image:
        if is_composite:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Target Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE (CRITICAL):
The reference image (image_0) is a **Composite Canvas** (a complex real-world photograph/screenshot containing the target asset).
Do NOT expect the candidate image under audit (image_1) to contain the surrounding objects, backgrounds, or full layout seen in image_0.
Instead, isolate the target subject (the logo, wordmark, UI component, or person the audit goal describes) inside image_0.
Verify if the candidate image (image_1) contains that target subject/style. Ignore all other background visual noise in image_0.
"""
        else:
            reference_instructions = f"""
You are given two images:
- The FIRST image (image_0) is the Reference Image (the template provided by the user).
- The SECOND image (image_1) is the Candidate Image (the asset under audit).

REFERENCE TEMPLATE DETAILS:
- Audit Goal: {audit_config.get('audit_goal')}
- Reference Image Description: {audit_config.get('image_description', 'No description')}

CANVAS ISOLATION DIRECTIVE:
Since the Reference Image (image_0) is a standalone logo, compare the candidate image (image_1) side-by-side against image_0.
The candidate matches even if the target appears as a SMALL element inside a larger composite/screenshot — a match anywhere on the canvas counts.
"""

    return f"""
Evaluate this image against the specified audit goal and criteria with high precision.

{reference_instructions}

[Match Mode Rule]:
{match_mode_rule}

[Color Independence Rule]:
Unless the Inclusion Criteria explicitly mention a required color, you must ignore any color differences between the reference image and the candidate image.

[Exact Typography Rule]:
Pay strict attention to typography and spelling (e.g. 'Google Play' is NOT 'Google Pay', 'Ads' is NOT 'AdWords'). Reject any assets that contain lookalike or misspelled branding unless the inclusion criteria explicitly permit them.

AUDIT SCOPE & EVALUATION RULES:

[Inclusion Criteria – Asset MUST fulfill these to pass]:
{inclusion_str}

[Exclusion Criteria – If asset triggers any of these, it MUST fail]:
{exclusion_str}

[Strict Adjudication Rule]:
{adjudication_logic}

[General Audit Instructions]:
{audit_instructions}

VERIFICATION STEPS (execute IN ORDER; record findings for each step in `visual_analysis_step_by_step`):
{steps_str}

FINAL EXECUTION:
1. Execute every VERIFICATION STEP above, scanning the entire canvas — a valid target match ANYWHERE on the candidate image counts, even as a small thumbnail inside a composite hero graphic.
2. Extract all diagnostic visual features requested in the schema, including whether the target is embedded inside a composite hero graphic (`is_embedded_in_composite_hero`).
3. Fill `criteria_verdicts` with one line per criterion (I1..In: PASS|FAIL, E1..En: TRIGGERED|CLEAR) citing the visual evidence.
4. Apply the Strict Adjudication Rule to your recorded evidence, then set `matches_criteria` to true only if the inclusion criteria are satisfied without triggering any exclusion criteria.
5. Provide a clear justification in `match_rationale` and an honest `match_confidence` (< 75 whenever occlusion, blur, or tiny scale creates doubt).
"""

async def run_llm_audit_single(asset_data: dict, audit_config: dict, _executor=None, reference_image_part: Optional[types.Part] = None, _model=None, _prompt=None) -> dict:
    """Evaluates a single image asset: hydration (GCS download + transparency blend) and Gemini inference
    are pipelined in one worker task, with transient-error retries. Schema/prompt are prebuilt per run."""
    DynamicAuditModel = _model or build_dynamic_audit_model(audit_config.get("extraction_schema", {}))
    audit_prompt = _prompt or build_audit_prompt(audit_config, reference_image_part is not None)
    gcs_path = asset_data["gcs_raw_path"]

    def _hydrate_and_infer():
        img_bytes, mime = _prep_image(_download_asset_bytes(gcs_path))
        image_part = types.Part.from_bytes(data=img_bytes, mime_type=mime)

        contents = []
        if reference_image_part:
            contents.append(reference_image_part)
        contents.append(image_part)
        contents.append(audit_prompt)

        return _generate_with_retry(lambda: client.models.generate_content(
            model=GEMINI_INFERENCE_MODEL,
            contents=contents,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=DynamicAuditModel,
                temperature=0.0
            )
        ))

    loop = asyncio.get_running_loop()
    try:
        response = await loop.run_in_executor(_executor, _hydrate_and_infer)
        extracted_data = json.loads(response.text)
    except Exception as e:
        extracted_data = {
            "matches_criteria": False,
            "match_confidence": 0,
            "match_rationale": f"Audit evaluation failed due to error: {str(e)}",
            "error": str(e)
        }

    # LLM output may never overwrite retrieval facts (paths, ids, scores); audit fields merge freely.
    extracted_data = {k: v for k, v in extracted_data.items() if k not in asset_data}
    return {**asset_data, **extracted_data}

async def run_llm_inference_on_dropoff_results(df_high: pd.DataFrame, df_edge: pd.DataFrame, audit_config: dict, max_workers: int = 30, reference_image_path: Optional[str] = None) -> pd.DataFrame:
    """Runs parallel multi-threaded LLM inference on candidate subsets, supporting side-by-side template
    matches, GCS/local transparency preprocessing, retries, and per-run prompt/schema caching."""
    candidates_df = pd.concat([df_high, df_edge], ignore_index=True)
    if candidates_df.empty:
        return pd.DataFrame()

    candidates = candidates_df.to_dict(orient="records")

    print(f"Starting parallel LLM audit inference on {len(candidates)} candidates (Max concurrency: {max_workers})...")

    # Helper to download and preprocess the reference image once
    def _get_reference_part():
        ref_bytes, mime = _prep_image(_download_asset_bytes(reference_image_path))
        return types.Part.from_bytes(data=ref_bytes, mime_type=mime)

    t0 = time.time()
    loop = asyncio.get_running_loop()

    total = len(candidates)
    progress = {"done": 0}
    progress_every = 100 if total > 400 else 0  # live progress + ETA for long Full Scan runs

    async def _run_with_progress(asset, executor, reference_image_part, shared_model, shared_prompt):
        res = await run_llm_audit_single(asset, audit_config, _executor=executor, reference_image_part=reference_image_part, _model=shared_model, _prompt=shared_prompt)
        progress["done"] += 1
        if progress_every and (progress["done"] % progress_every == 0 or progress["done"] == total):
            elapsed = time.time() - t0
            rate = progress["done"] / max(elapsed, 1e-6)
            eta_min = (total - progress["done"]) / max(rate, 1e-6) / 60.0
            print(f" Progress: {progress['done']}/{total} audited | {rate:.2f} img/s | ETA ~{eta_min:.0f} min")
        return res

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        reference_image_part = None
        if reference_image_path:
            reference_image_part = await loop.run_in_executor(executor, _get_reference_part)

        # Build the structured-output schema and audit prompt ONCE for the whole run.
        shared_model = build_dynamic_audit_model(audit_config.get("extraction_schema", {}))
        shared_prompt = build_audit_prompt(audit_config, reference_image_part is not None)

        tasks = [
            _run_with_progress(asset, executor, reference_image_part, shared_model, shared_prompt)
            for asset in candidates
        ]
        results = await asyncio.gather(*tasks)

    duration = time.time() - t0
    print(f" Processing assets... [{len(candidates)}/{len(candidates)}] completed in {duration:.1f}s")
    print(f" Inference completed. Enforcing Zero-FP policies and sorting scoreboard...")

    results_df = pd.DataFrame(results)
    results_df = enforce_zero_false_positives_rules(results_df)

    if "relevance_score" in results_df.columns:
        results_df = results_df.sort_values(by="relevance_score", ascending=False).reset_index(drop=True)

    SCOREBOARD_MAX_ROWS = 200
    n_pass = int((results_df["matches_criteria"] == True).sum()) if "matches_criteria" in results_df.columns else 0
    print(f"\n=== VERIFIED AUDIT SCOREBOARD (Sorted by Search Similarity) — {n_pass} PASS / {len(results_df)} audited ===")
    for i, r in results_df.head(SCOREBOARD_MAX_ROWS).iterrows():
        status_label = "PASS" if r.get("matches_criteria", False) else "FAIL"
        filename = r.get("asset_filename") or r.get("gcs_raw_path", "").split("/")[-1]
        sim = r.get("relevance_score", 0.0)
        conf = r.get("match_confidence", 100)
        print(f"[{i+1}/{len(results_df)}] {status_label} | {str(filename)[:40]} | Sim: {sim:.4f} | Conf: {conf}%")
    if len(results_df) > SCOREBOARD_MAX_ROWS:
        print(f" ... and {len(results_df) - SCOREBOARD_MAX_ROWS} more rows (full scoreboard in the results CSV).")

    return results_df

In [ ]:
# 4. Calibration Summary & Audit Results Saving E2E Loop
async def save_audit_results_to_db(session_id: str, results_df: pd.DataFrame):
    """Saves the final audited results into AlloyDB (Bypassed by default in the interactive runner)."""
    if results_df.empty:
        return

    engine, _ = await get_alloydb_connection()
    async with engine.connect() as conn:
        raw_conn = await conn.get_raw_connection()
        db = raw_conn.driver_connection
        records = results_df.to_dict("records")
        for r in records:
            verdict = "PASS" if r.get("matches_criteria") is True else "FAIL"

            # Extract criteria_checks from JSON response or construct it
            criteria_checks = {k: v for k, v in r.items() if k not in ["asset_id", "gcs_raw_path", "matches_criteria", "error", "asset_filename", "page_url"]}

            await db.execute(
                f"""
                INSERT INTO {DB_SCHEMA}.audit_results (
                    session_id, asset_id, overall_verdict, adjudication_result,
                    criteria_checks, rationale, confidence_band
                ) VALUES ($1, $2, $3, $4, $5, $6, $7)
                """,
                session_id,
                r.get("asset_id"),
                verdict,
                r.get("matches_criteria", False),
                json.dumps(criteria_checks),
                r.get("match_rationale", "Completed"),
                "high" if r.get("match_confidence", 0) > 95 else ("borderline" if r.get("match_confidence", 0) >= 70 else "below_threshold")
            )
    print("Audit results saved to database.")

async def generate_ai_audit_summary(results_df: pd.DataFrame, audit_config: dict) -> str:
    """Generates an AI-powered executive summary of the visual asset audit results."""
    import json
    import numpy as np
    if results_df is None or results_df.empty:
        return "No audit results available to generate a summary."

    # Select columns to pass to the LLM, avoiding internal or verbose vector columns
    exclude_cols = {"num_chunks", "max_relevance_score", "gcs_raw_path", "gcs_processed_path", "embedding", "embedding_at"}
    cols_to_include = [col for col in results_df.columns if col not in exclude_cols]

    # Convert to records safely, handling NumPy arrays, lists, and floats without boolean truth value ambiguity
    clean_df = results_df[cols_to_include].copy()
    for col in clean_df.columns:
        def safe_clean(val):
            if val is None:
                return None
            if isinstance(val, (np.ndarray, pd.Series)):
                return val.tolist() if val.size > 0 else None
            if isinstance(val, float) and pd.isna(val):
                return None
            return val
        clean_df[col] = clean_df[col].apply(safe_clean)

    records = clean_df.to_dict(orient="records")
    formatted_results = json.dumps(records, indent=2)

    audit_instructions = audit_config.get("audit_instructions", "No specific audit context provided.")

    # Compute basic stats to seed in the prompt
    total_audited = len(results_df)
    matches_col = "matches_criteria" if "matches_criteria" in results_df.columns else None
    if matches_col:
        # Convert to boolean safely, handling string representation if any
        matches_true = results_df[matches_col].apply(lambda x: str(x).lower() in ("true", "1", "yes")).sum()
    else:
        matches_true = "N/A"

    summary_prompt = f"""
You are a Lead Visual Asset Auditor.
Your task is to write a visually engaging, highly structured, and extremely concise summary of a visual asset audit Test Bench calibration run.

STRICT RULES FOR FORMATTING & SECTIONS:
1. You MUST only include the following exact three sections in the output:
   - Objective & Scope (Preview Subset) (within the top [!NOTE] block)
   - Calibration Statistics (as a numbered list)
   - Configuration Calibration Insights (as a single, brief narrative paragraph of 3-4 sentences detailing the visual rules performance)
2. DO NOT include any other sections.
3. DO NOT pass any definitive verdicts of success.

AUDIT CONTEXT (Visual Evaluation Criteria):
{audit_instructions}

TEST BENCH STATS:
- Total images audited: {total_audited}
- Total images matching criteria (True): {matches_true}

TEST BENCH FINDINGS (JSON):
{formatted_results}
"""

    response = client.models.generate_content(
        model=GEMINI_ORCHESTRATOR_MODEL,
        contents=summary_prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text


## Telemetry Summaries & Calibration Database Saving

This cell defines functions to generate final executive AI summaries (`generate_ai_audit_summary`) and save audit session details into the run calibration tables in AlloyDB for telemetry history.

In [ ]:
# 5. E2E Execution Helpers (Interactive Split Stages)
import asyncio
from typing import Optional
import base64
import os
import time
import json
import pandas as pd
from google.cloud import storage

def get_gcs_image_base64(gcs_path: str) -> str:
    """Downloads image from GCS or local file and returns its base64 data URI for inline HTML rendering."""
    try:
        image_bytes = b""
        mime_type = "image/png"

        # Detect format
        lower_path = gcs_path.lower()
        if lower_path.endswith(".jpg") or lower_path.endswith(".jpeg"):
            mime_type = "image/jpeg"
        elif lower_path.endswith(".webp"):
            mime_type = "image/webp"
        elif lower_path.endswith(".gif"):
            mime_type = "image/gif"

        if gcs_path.startswith("gs://"):
            parts = gcs_path.replace("gs://", "").split("/", 1)
            bucket_name = parts[0]
            blob_name = parts[1]

            # Authenticated download using the cached, project-bound client (defined in the inference cell)
            storage_client = _get_storage_client()
            bucket = storage_client.bucket(bucket_name)
            blob = bucket.blob(blob_name)
            image_bytes = blob.download_as_bytes()
        elif os.path.exists(gcs_path):
            with open(gcs_path, "rb") as f:
                image_bytes = f.read()
        else:
            return ""

        encoded = base64.b64encode(image_bytes).decode("utf-8")
        return f"data:{mime_type};base64,{encoded}"
    except Exception as e:
        return ""

async def generate_audit_config_only(user_goal: str, reference_image_path: Optional[str] = None) -> dict:
    """Stage 1: SINGLE fused forensic + config LLM call — the reference image is attached directly to the
    config-builder call, which writes the forensic breakdown into `image_description` itself. The sampled
    tag-vocabulary fetch runs CONCURRENTLY with the LLM call (session-cached; no full-table DISTINCT scan)
    and the proposed tags are grounded against the real vocabulary afterwards, so the prompt never carries
    the tag list (token-overflow fix)."""
    t0 = time.time()

    # Kick off the (session-cached) sampled vocabulary fetch so it overlaps the LLM call.
    vocab_task = asyncio.create_task(get_relevant_tag_vocabulary())

    if reference_image_path:
        print("1. Generating Audit Configuration (fused forensic analysis + rules in ONE call)...")
    else:
        print("1. Translating goal into Audit Configuration...")

    loop = asyncio.get_running_loop()
    audit_config = await loop.run_in_executor(
        None,
        lambda: generate_audit_config(user_goal, reference_image_description=None, available_tags=None, reference_image_path=reference_image_path)
    )

    if reference_image_path and audit_config.get("image_description"):
        print(f"Forensic Reference Image Analysis (fused):\n{audit_config['image_description']}\n{'='*50}")

    # Ground the proposed tags against the real database vocabulary (sampled, cached per session).
    vocabulary = await vocab_task
    proposed_tags = audit_config.get("vision_tag_filter", [])
    audit_config["vision_tag_filter"] = ground_tags_against_vocabulary(proposed_tags, audit_config.get("search_keywords", []), vocabulary)
    if vocabulary:
        print(f"Vision tags grounded against sampled vocabulary ({len(vocabulary)} tags): {proposed_tags} -> {audit_config['vision_tag_filter']}")

    print(f"Generated Configuration in {time.time() - t0:.2f}s:\n", json.dumps(audit_config, indent=2, default=str))
    return audit_config

QUICK_MODE_INFERENCE_CAP = 40      # Quick mode: visually audit only the top-N reranked candidates
QUICK_MODE_RETRIEVAL_LIMIT = 1000  # Quick mode: bounded fused candidate pool from hybrid search
FULL_MODE_RETRIEVAL_LIMIT = 100000 # Full Scan mode: retrieval arms may scan up to 1 lakh rows
RENDER_IMAGE_MAX_ROWS = 60         # inline base64 previews only for small, non-quick result sets
DISPLAY_MAX_ROWS = 200             # cap the inline HTML results table (full results live in the CSV)
SUMMARY_MAX_ROWS = 150             # cap rows fed to the executive-summary LLM (token-overflow guard)

def save_results_csv(results_df: pd.DataFrame, mode_label: str) -> Optional[str]:
    """Writes the full audit scoreboard to a CSV with complete, untruncated asset URLs."""
    if results_df is None or results_df.empty:
        return None
    try:
        csv_path = os.path.abspath(f"audit_results_{mode_label}_{time.strftime('%Y%m%d_%H%M%S')}.csv")
        csv_df = results_df.copy()
        preferred = [c for c in ["asset_id", "gcs_raw_path", "page_url", "asset_filename", "matches_criteria",
                                 "match_confidence", "match_rationale", "criteria_verdicts", "relevance_score",
                                 "cross_encoder_score", "vector_distance", "gemini_description",
                                 "visual_analysis_step_by_step"] if c in csv_df.columns]
        csv_df = csv_df[preferred + [c for c in csv_df.columns if c not in preferred]]
        csv_df.to_csv(csv_path, index=False)
        print(f"💾 Audit results CSV saved: {csv_path} ({len(csv_df)} rows, full asset URLs included)")
        return csv_path
    except Exception as e:
        print(f"Warning: could not save results CSV ({e})")
        return None

async def run_full_test_bench_pipeline_execution(audit_config: dict, reference_image_path: Optional[str] = None, quick_mode: bool = True) -> pd.DataFrame:
    """Stage 2, 3, 3.5, 4, 5: Executes Retrieval, Segmentation, Reranking, Inference, and Calibration scorecards.

    quick_mode=True  (Quick Scan): bounded hybrid search (fast per-arm caps), Kneedle + batched
        cross-encoder, then visual inference on only the TOP 40 candidates (QUICK_MODE_INFERENCE_CAP).
    quick_mode=False (Full Scan): hybrid search up to 1 lakh rows per text arm; the text cross-encoder
        is skipped and EVERY unique retrieved candidate goes to visual inference.
    Both modes save the complete scoreboard to a CSV with full asset URLs."""
    pipeline_start_time = time.time()
    telemetry = {}
    mode_label = "quick" if quick_mode else "full"

    if quick_mode:
        print(f"⚙️ MODE: QUICK SCAN — bounded retrieval, top {QUICK_MODE_INFERENCE_CAP} visual audits, CSV with full asset URLs. (Pass quick_mode=False for the exhaustive Full Scan.)")
    else:
        print("⚙️ MODE: FULL SCAN — retrieval up to 1 lakh rows, EVERY unique candidate visually audited. (Pass quick_mode=True for the fast bounded scan.)")

    t0 = time.time()
    retrieval_limit = QUICK_MODE_RETRIEVAL_LIMIT if quick_mode else FULL_MODE_RETRIEVAL_LIMIT
    print(f"\n2. Running Parallel Hybrid Search with RRF ({'Quick' if quick_mode else 'Full Scan'} mode, retrieval limit {retrieval_limit:,})...")
    search_results = await run_hybrid_search({}, audit_config, reference_image_path, limit=retrieval_limit, quick_mode=quick_mode)
    telemetry["Stage 2 (Multi-Arm RRF Retrieval)"] = f"{time.time() - t0:.2f}s | {len(search_results)} unique candidates retrieved"
    print(f"Found {len(search_results)} unique candidates.")

    t0 = time.time()
    print("\n3. Running Drop-off Analysis (Kneedle + Volatility)...")
    df_results = pd.DataFrame(search_results)
    df_high, df_edge, df_low = detect_dropoff_flawless(df_results)
    telemetry["Stage 3 (Drop-off Segmentation)"] = f"{time.time() - t0:.2f}s | High: {len(df_high)}, Borderline: {len(df_edge)}, Low: {len(df_low)}"
    print(f"Rough Candidates -> High: {len(df_high)} | Borderline: {len(df_edge)} | Low: {len(df_low)}")

    # Save full results globally for evaluation recall calculation
    globals()["df_results_full"] = df_results

    if quick_mode:
        t0 = time.time()
        print("\n3.5. Running Semantic Reranking and Fine Filtering...")
        df_high_sem, df_edge_sem, df_low_sem = await run_semantic_reranking_and_filter(df_high, df_edge, audit_config)
        telemetry["Stage 3.5 (Semantic Reranking)"] = f"{time.time() - t0:.2f}s | High: {len(df_high_sem)}, Borderline: {len(df_edge_sem)}, Demoted: {len(df_low_sem)}"
        combined = pd.concat([df_high_sem, df_edge_sem], ignore_index=True)
        if not combined.empty and "relevance_score" in combined.columns:
            combined = combined.sort_values(by="relevance_score", ascending=False)
        candidates_high = combined.head(QUICK_MODE_INFERENCE_CAP).reset_index(drop=True)
        candidates_edge = pd.DataFrame()
        print(f"\n Quick Mode: visually auditing the top {len(candidates_high)} of {len(combined)} reranked candidates.")
    else:
        print("\n3.5. Full Scan mode: skipping text cross-encoder — EVERY unique retrieved candidate goes to visual inference.")
        telemetry["Stage 3.5 (Semantic Reranking)"] = "skipped (Full Scan audits all unique candidates)"
        candidates_high = df_results
        candidates_edge = pd.DataFrame()
        est_minutes = len(df_results) * 4.0 / 30 / 60.0
        print(f" Full Scan will visually audit {len(df_results):,} unique assets (rough estimate ~{est_minutes:.0f} min at 30 workers / ~4s per image).")

    t0 = time.time()
    print("\n4. Running Parallel LLM Audit Inference...")
    results_df = await run_llm_inference_on_dropoff_results(candidates_high, candidates_edge, audit_config, reference_image_path=reference_image_path)
    inf_duration = time.time() - t0
    throughput = len(results_df) / inf_duration if inf_duration > 0 else 0
    telemetry["Stage 4 (Parallel Visual Inference)"] = f"{inf_duration:.2f}s | Throughput: {throughput:.2f} images/sec"
    print(f"LLM Results: {len(results_df)} assets audited.")

    # Persist the complete scoreboard (full asset URLs, no truncation) before anything else can fail.
    csv_path = save_results_csv(results_df, mode_label)

    t0 = time.time()
    if len(results_df) > SUMMARY_MAX_ROWS:
        if "matches_criteria" in results_df.columns:
            pass_rows = results_df[results_df["matches_criteria"] == True].head(100)
        else:
            pass_rows = results_df.head(0)
        summary_input = pd.concat([pass_rows, results_df.head(SUMMARY_MAX_ROWS)], ignore_index=True)
        if "asset_id" in summary_input.columns:
            summary_input = summary_input.drop_duplicates(subset=["asset_id"])
        summary_input = summary_input.head(SUMMARY_MAX_ROWS).reset_index(drop=True)
        print(f"\n5. Generating Calibration Summary (from a {len(summary_input)}-row sample of {len(results_df)} results)...")
    else:
        summary_input = results_df.copy()
        print("\n5. Generating Calibration Summary...")
    if "asset_id" in summary_input.columns:
        summary_input["asset_id"] = summary_input["asset_id"].astype(str)  # uuid.UUID would break cell 12's json.dumps
    summary = await generate_ai_audit_summary(summary_input, audit_config)
    telemetry["Stage 5 (Executive Summary Generation)"] = f"{time.time() - t0:.2f}s"

    total_time = time.time() - pipeline_start_time

    # Compute Observability Stats
    error_count = results_df["error"].notna().sum() if (not results_df.empty and "error" in results_df.columns) else 0
    high_conf = (results_df["match_confidence"] >= 95).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0
    borderline_conf = ((results_df["match_confidence"] >= 70) & (results_df["match_confidence"] < 95)).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0
    low_conf = (results_df["match_confidence"] < 70).sum() if (not results_df.empty and "match_confidence" in results_df.columns) else 0

    try:
        from IPython.display import display, Markdown, HTML

        telemetry_md = f"""
### 📊 Enterprise Observability & Telemetry Scorecard ({'Quick Scan' if quick_mode else 'Full Scan'})
| Stage / Metric | Value / Duration | Status |
|:--- |:--- |:---|
| **Total Pipeline Execution Time** | **{total_time:.2f}s** | 🟢 Optimal Throughput |
| **Parallel Inference Speed** | **{throughput:.2f} images/sec** | Concurrency: 30 Workers |
| **Stage 2: Multi-Arm RRF Retrieval** | {telemetry.get('Stage 2 (Multi-Arm RRF Retrieval)', 'N/A')} | ANN + GIN Index-Bound Arms |
| **Stage 3: Kneedle Segmentation** | {telemetry.get('Stage 3 (Drop-off Segmentation)', 'N/A')} | Noise Tail Truncated |
| **Stage 3.5: Semantic Reranking** | {telemetry.get('Stage 3.5 (Semantic Reranking)', 'N/A')} | Batched LLM Text Cross-Encoder |
| **Stage 4: Vision Inference** | {telemetry.get('Stage 4 (Parallel Visual Inference)', 'N/A')} | 0 False Positive Verdict Guarantee |
| **Results CSV** | {csv_path or 'not saved'} | Full asset URLs |
| **Confidence Band Distribution** | High (>95%): **{high_conf}** || Borderline (70-95%): **{borderline_conf}** || Low (<70%): **{low_conf}** || Error Count: **{error_count}** |
"""
        display(Markdown(telemetry_md))
        display(Markdown(summary))

        if not results_df.empty:
            table_note = f" (top {DISPLAY_MAX_ROWS} of {len(results_df)} — full results in the CSV)" if len(results_df) > DISPLAY_MAX_ROWS else ""
            display(Markdown("### Detailed Audit Results" + table_note))
            display_df = results_df.head(DISPLAY_MAX_ROWS).copy()

            render_images = (not quick_mode) and len(display_df) <= RENDER_IMAGE_MAX_ROWS
            cols, col_labels = [], []
            if render_images:
                display_df["Visual Preview"] = display_df["gcs_raw_path"].apply(
                    lambda x: (lambda b64: f'<img src="{b64}" width="150" />' if b64 else '[No Preview]')(get_gcs_image_base64(x))
                )
                cols.append("Visual Preview")
                col_labels.append("Visual Preview")

            display_df["Page Link"] = display_df["page_url"].apply(
                lambda x: f'<a href="{x}" target="_blank">{x}</a>' if x else '[No Page Link]'
            )
            cols += ["matches_criteria", "match_confidence", "relevance_score", "gcs_raw_path", "Page Link", "match_rationale"]
            col_labels += ["Matches Criteria", "Confidence", "Search Similarity", "Asset URL (full)", "Page Location URL", "Rationale"]

            if "gemini_description" in display_df.columns:
                cols.append("gemini_description")
                col_labels.append("Image Description")

            display_df = display_df[cols]
            display_df.columns = col_labels

            with pd.option_context('display.max_colwidth', None):
                display(HTML(display_df.to_html(escape=False, index=False)))
    except (ImportError, ModuleNotFoundError):
        print("\n=== TELEMETRY SCORECARD ===")
        for k, v in telemetry.items():
            print(f"  {k}: {v}")
        print(f"  Total Time: {total_time:.2f}s | Errors: {error_count}")
        if csv_path:
            print(f"  Results CSV: {csv_path}")
        print("\n=== CALIBRATION SUMMARY ===")
        print(summary)

    return results_df

# Bypassed original monolithic E2E pipeline name to map to partitioned runners
async def run_full_test_bench_pipeline(user_goal: str, reference_image_path: Optional[str] = None, quick_mode: bool = True) -> pd.DataFrame:
    """Wrapper to maintain backwards compatibility for existing cells."""
    config = await generate_audit_config_only(user_goal, reference_image_path)
    return await run_full_test_bench_pipeline_execution(config, reference_image_path, quick_mode=quick_mode)

## E2E Execution & Telemetry Helpers

This cell defines the core pipeline orchestrators: `generate_audit_config_only` (Stage 1 Config) and `run_full_test_bench_pipeline_execution` (Stage 2 E2E). It coordinates retrieval, segmentation, cross-encoder reranking, and visual LLM inference, while logging telemetry and formatted scorecard widgets.

In [ ]:
# TEST RUN (Stage 1): Generate Audit Rules & Configuration
# Run this cell to upload your reference image and generate the rule configuration.
import nest_asyncio
nest_asyncio.apply()

# Dynamic Reference Image Detector (Triggered via Colab Interactive Upload)
reference_image_path = None
try:
    from google.colab import files
    print("[OPTIONAL] Upload a reference image for comparative compliance audit:")
    uploaded = files.upload()
    if uploaded:
        reference_image_path = list(uploaded.keys())[0]
        print(f" Reference image uploaded: {reference_image_path}")
    else:
        print(" No reference image uploaded. Running text-only audit goal.")
except Exception as e:
    reference_image_path = None

# Enterprise Audit Goal (Modify this as needed)
TEST_AUDIT_GOAL = "Find all pages with this exact image"

# Step 1: Generate configuration rules
audit_config_global = None
try:
    audit_config_global = await generate_audit_config_only(
        user_goal=TEST_AUDIT_GOAL,
        reference_image_path=reference_image_path
    )
    print("💡 TIP: You can inspect and tweak 'audit_config_global' directly in the cell below before running Stage 2.")
except Exception as e:
    print(f"Error generating audit configuration: {e}")


In [ ]:
# TEST RUN (Stage 2): Execute Visual Search & Parallel Audit
# Run this cell to execute retrieval, segmentation, and LLM inference.
# You can uncomment and modify rules below to calibrate config before executing.

if 'audit_config_global' in globals() and audit_config_global is not None:
    # OPTIONAL CALIBRATION TUNING:
    # If the generated AI rules were slightly off, you can uncomment and edit them here:
    # audit_config_global["inclusion_criteria"] = [
    #     "The image must contain the legacy Google Pay logo featuring the interlocking loops design."
    # ]
    # audit_config_global["exclusion_criteria"] = [
    #     "Exclude images containing the current compliant Google Pay GPay wordmark button logo."
    # ]

    df_results_global = None
    try:
        df_results_global = await run_full_test_bench_pipeline_execution(
            audit_config=audit_config_global,
            reference_image_path=reference_image_path
        )
    except Exception as e:
        print(f"Error executing test bench pipeline: {e}")
else:
    print(" Please run Stage 1 cell first to generate 'audit_config_global'.")
